# Rough Heston FNO — Training Notebook

Trains the Fourier Neural Operator to predict the Hurst exponent H(t) from
synthetic rough Heston paths.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (or A100 if available on Colab Pro)
2. Upload the five source files from `colab_training/` using Cell 3,
   OR mount your Google Drive if you have synced the repo there.

**Output:** `checkpoints/fno_best.pt` — download this and place it in
`models/checkpoints/` in your local repo.

## 1 · GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅  GPU: {gpu}  ({mem:.1f} GB VRAM)")
else:
    print("⚠️  No GPU detected — go to Runtime → Change runtime type → GPU")

## 2 · Install dependencies

In [ ]:
# PyTorch is pre-installed on Colab. Install the remaining packages.
!pip install -q numpy scipy pandas pyarrow tqdm

## 3 · Upload source files

Upload the five files from the `colab_training/` folder in your repo:
`config.py`, `utils.py`, `spde_simulator.py`, `fno.py`, `trainer.py`

**Option A — Manual upload (simplest)**

In [ ]:
from google.colab import files

print("Select the 5 .py files from colab_training/ on your computer:")
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
import os

required = ["config.py", "utils.py", "spde_simulator.py", "fno.py", "trainer.py"]
missing  = [f for f in required if not os.path.exists(f)]
if missing:
    print(f"⚠️  Missing files: {missing}")
else:
    print("✅  All source files present.")

**Option B — Google Drive mount** (use instead of Option A if your repo is in Drive)

In [ ]:
# Uncomment and edit the path if using Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil, os
# SRC = "/content/drive/MyDrive/Rough-Heston-Trading-Model/colab_training"
# for f in ["config.py", "utils.py", "spde_simulator.py", "fno.py", "trainer.py"]:
#     shutil.copy(f"{SRC}/{f}", f"/content/{f}")
# print("Copied source files from Drive.")

## 4 · Configuration

Adjust these to trade off training time vs. data volume.
The defaults below are tuned for a Colab T4 (16 GB VRAM).

In [ ]:
import config

# ── Data generation ──────────────────────────────────────────────────────────
# 2000 paths per H value → 18,000 paths total → ~45 min simulation on CPU
# Reduce to 500 for a quick smoke-test (~10 min)
config.N_PATHS_PER_HURST = 2_000

# ── FNO architecture (leave as-is unless you want to experiment) ─────────────
config.FNO_WIDTH  = 64
config.FNO_MODES  = 16
config.FNO_LAYERS = 4

# ── Training ─────────────────────────────────────────────────────────────────
config.NUM_EPOCHS              = 200
config.BATCH_SIZE              = 128   # larger batch fits comfortably on T4
config.LEARNING_RATE           = 1e-3
config.EARLY_STOPPING_PATIENCE = 20

# ── Workers (Colab has 2 CPU cores by default) ───────────────────────────────
config.NUM_WORKERS = 2

print("Config:")
print(f"  Paths per H value : {config.N_PATHS_PER_HURST}")
print(f"  Total paths       : {len(config.HURST_GRID) * config.N_PATHS_PER_HURST:,}")
print(f"  Epochs (max)      : {config.NUM_EPOCHS}")
print(f"  Batch size        : {config.BATCH_SIZE}")

## 5 · Generate synthetic training data

Simulates rough Heston paths across the Hurst grid and saves them to parquet.
If you already ran this and the file exists, it loads from cache automatically.

In [ ]:
from spde_simulator import SPDESimulator

sim = SPDESimulator(
    n_paths_per_hurst=config.N_PATHS_PER_HURST,
    num_workers=config.NUM_WORKERS,
)
dataset_df = sim.generate_dataset(save=True)

print(f"\nDataset shape : {dataset_df.shape}")
print(f"H values      : {sorted(dataset_df['H'].unique())}")
print(f"Paths         : {dataset_df['path_id'].nunique():,}")
dataset_df.head()

## 6 · Train the FNO

In [ ]:
from trainer import FNOTrainer

trainer = FNOTrainer(
    num_epochs=config.NUM_EPOCHS,
    batch_size=config.BATCH_SIZE,
    learning_rate=config.LEARNING_RATE,
    patience=config.EARLY_STOPPING_PATIENCE,
)

history = trainer.train(dataset_df)

print(f"\nBest epoch    : {history.best_epoch + 1}")
print(f"Best val MSE  : {history.best_val_loss:.6f}")
print(f"Best val MAE  : {history.val_maes[history.best_epoch]:.4f}")
print(f"Checkpoint    : {history.checkpoint_path}")

## 7 · Training curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history.train_losses) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# MSE
axes[0].plot(epochs, history.train_losses, label="Train MSE", linewidth=1.5)
axes[0].plot(epochs, history.val_losses,   label="Val MSE",   linewidth=1.5)
axes[0].axvline(history.best_epoch + 1, color="red", linestyle="--", alpha=0.6,
                label=f"Best epoch {history.best_epoch + 1}")
axes[0].set_title("MSE Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(epochs, history.val_maes, color="green", label="Val MAE", linewidth=1.5)
axes[1].axvline(history.best_epoch + 1, color="red", linestyle="--", alpha=0.6,
                label=f"Best epoch {history.best_epoch + 1}")
axes[1].set_title("Validation MAE (H prediction error)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("FNO Training — Rough Heston Hurst Exponent Prediction", fontsize=13)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curves.png")

## 8 · Evaluation — predicted vs. true H on held-out paths

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# Load the best checkpoint
trainer.load_checkpoint(history.checkpoint_path)
model  = trainer.model
device = trainer.device
model.eval()

W = config.WINDOW_SIZE
true_H, pred_H = [], []

for h_val, grp in dataset_df.groupby("H"):
    # Take the first path for this H value
    first_pid = grp["path_id"].iloc[0]
    path  = grp[grp["path_id"] == first_pid].sort_values("step")
    feats = path[["log_return", "variance"]].to_numpy(dtype=np.float32)
    if len(feats) < W:
        continue
    window = feats[:W].copy()
    mu  = window.mean(axis=0, keepdims=True)
    std = window.std(axis=0, keepdims=True) + 1e-8
    window = (window - mu) / std
    inp = torch.from_numpy(window.T[np.newaxis]).to(device)  # (1, 2, W)
    with torch.no_grad():
        h_pred = float(model(inp).squeeze().cpu())
    true_H.append(h_val)
    pred_H.append(h_pred)

true_H = np.array(true_H)
pred_H = np.array(pred_H)
mae    = np.abs(true_H - pred_H).mean()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(true_H, pred_H, s=80, zorder=3)
lims = [true_H.min() - 0.02, true_H.max() + 0.02]
ax.plot(lims, lims, "r--", linewidth=1, label="Perfect prediction")
for th, ph in zip(true_H, pred_H):
    ax.annotate(f"{th:.2f}", (th, ph), textcoords="offset points",
                xytext=(5, 3), fontsize=8)
ax.set_xlabel("True H")
ax.set_ylabel("Predicted H")
ax.set_title(f"FNO: Predicted vs. True Hurst  (MAE = {mae:.4f})")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("pred_vs_true_H.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Evaluation MAE: {mae:.4f}")

## 9 · Rolling H prediction on a sample path

Shows how H(t) evolves over a single simulated path — this is what the
strategy uses to switch regimes in real time.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from spde_simulator import SPDESimulator

# Simulate a single 504-day path (2 years) with H = 0.25 (rough)
sim_single = SPDESimulator(T=504)
path_dict  = sim_single.simulate_path(H=0.25, seed=1)

returns  = pd.Series(path_dict["log_returns"], name="log_return")
variance = pd.Series(path_dict["variance"],    name="variance")

# Rolling H inference
feats  = np.stack([returns.values, variance.values], axis=1).astype(np.float32)
T_len  = len(feats)
W      = config.WINDOW_SIZE
h_vals = np.full(T_len, np.nan)

model.eval()
with torch.no_grad():
    for i in range(W, T_len + 1):
        win = feats[i - W: i].copy()
        mu  = win.mean(axis=0, keepdims=True)
        std = win.std(axis=0, keepdims=True) + 1e-8
        win = (win - mu) / std
        inp = torch.from_numpy(win.T[np.newaxis]).to(device)
        h_vals[i - 1] = float(model(inp).squeeze().cpu())

H_rolling = pd.Series(h_vals, name="H_fno")

# ── Plot ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)

axes[0].plot(returns.values, linewidth=0.8, color="steelblue")
axes[0].set_ylabel("Log return")
axes[0].set_title("Simulated rough Heston path  (true H = 0.25)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(variance.values, linewidth=0.8, color="orange")
axes[1].set_ylabel("Variance V(t)")
axes[1].grid(True, alpha=0.3)

axes[2].plot(H_rolling.values, linewidth=1.2, color="purple", label="FNO H(t)")
axes[2].axhline(0.25, color="red",  linestyle="--", alpha=0.7, label="True H = 0.25")
axes[2].axhline(0.35, color="gray", linestyle=":",  alpha=0.5, label="Mean-rev threshold")
axes[2].axhline(0.65, color="gray", linestyle=":",  alpha=0.5, label="Momentum threshold")
h_filled = H_rolling.fillna(0.5).values
axes[2].fill_between(range(T_len), h_filled, 0.35,
                     where=h_filled < 0.35,
                     alpha=0.15, color="blue", label="Mean-revert zone")
axes[2].fill_between(range(T_len), h_filled, 0.65,
                     where=h_filled > 0.65,
                     alpha=0.15, color="green", label="Momentum zone")
axes[2].set_ylabel("Predicted H")
axes[2].set_xlabel("Time step (days)")
axes[2].set_ylim(0, 1)
axes[2].legend(fontsize=8, loc="upper right")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("rolling_H_prediction.png", dpi=150, bbox_inches="tight")
plt.show()

## 10 · Download the checkpoint and plots

Download `fno_best.pt` to your local machine, then copy it to
`models/checkpoints/fno_best.pt` in your repo.
The plots can go into `results/plots/` for your paper.

In [ ]:
from google.colab import files

files.download(str(history.checkpoint_path))   # fno_best.pt
files.download("training_curves.png")
files.download("pred_vs_true_H.png")
files.download("rolling_H_prediction.png")

## Summary

| Item | Value |
|------|-------|
| Best epoch | *(filled after training)* |
| Best val MSE | *(filled after training)* |
| Best val MAE | *(filled after training)* |

**Next steps (back on your local machine):**
1. Copy `fno_best.pt` → `models/checkpoints/fno_best.pt`
2. Run the full backtest pipeline with `mode='fno'`:
   ```python
   from wsq_trading.pipeline import Pipeline
   from pathlib import Path
   pipe = Pipeline(mode='fno', fno_checkpoint=Path('models/checkpoints/fno_best.pt'))
   results, weights, summary = pipe.run_and_summarise()
   print(summary)
   ```